In [1]:
# 1. Import Libraries

import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

In [7]:
# 2. Load Cleaned Data

data = pd.read_csv('cleaned_chip_data.csv')

# Convert DATE
data['DATE'] = pd.to_datetime(data['DATE'])

# Create Year-Month for aggregation
data['YEARMONTH'] = data['DATE'].dt.to_period('M')

In [10]:
# 3. Create Store-Level Metrics

store_metrics = data.groupby(['STORE_NBR', 'YEARMONTH']).agg(
    TOTAL_SALES=('TOT_SALES', 'sum'),
    N_CUSTOMERS=('LYLTY_CARD_NBR', 'nunique'),
    N_TXNS=('TXN_ID', 'nunique')
).reset_index()

# Derived metric
store_metrics['TXNS_PER_CUSTOMER'] = store_metrics['N_TXNS'] / store_metrics['N_CUSTOMERS']

In [11]:
# 4. Define Trial Period

trial_start = '2019-02'
trial_end = '2019-04'

# Pre-trial period
pre_trial = store_metrics[store_metrics['YEARMONTH'] < trial_start]

In [12]:
# 5. Function to Find Control Store

def find_control_store(trial_store, metric_col):
    trial_data = pre_trial[pre_trial['STORE_NBR'] == trial_store]
    correlations = []
    for store in pre_trial['STORE_NBR'].unique():
        if store == trial_store:
            continue
        control_data = pre_trial[pre_trial['STORE_NBR'] == store]
        merged = pd.merge(trial_data, control_data, 
                          on='YEARMONTH', suffixes=('_trial', '_control'))
        if len(merged) > 1:
            corr = merged[f'{metric_col}_trial'].corr(merged[f'{metric_col}_control'])
            diff = np.mean(np.abs(
                merged[f'{metric_col}_trial'] - merged[f'{metric_col}_control']
            ))
            correlations.append({
                'STORE_NBR': store,
                'CORRELATION': corr,
                'MAG_DISTANCE': diff
            })
    df = pd.DataFrame(correlations)
    # Normalize distance → similarity score
    df['DIST_SCORE'] = 1 - (
        (df['MAG_DISTANCE'] - df['MAG_DISTANCE'].min()) /
        (df['MAG_DISTANCE'].max() - df['MAG_DISTANCE'].min())
    )
    # Final score
    df['FINAL_SCORE'] = df['CORRELATION'] * 0.5 + df['DIST_SCORE'] * 0.5
    return df.sort_values('FINAL_SCORE', ascending=False).iloc[0]['STORE_NBR']

In [13]:
# 6. Identify Control Stores

trial_stores = [77, 86, 88]
control_stores = {}

for store in trial_stores:
    control = find_control_store(store, 'TOTAL_SALES')
    control_stores[store] = control

print("Control Stores:", control_stores)

Control Stores: {77: np.float64(233.0), 86: np.float64(155.0), 88: np.float64(125.0)}


In [14]:
# 7. Compare Trial vs Control

def compare_trial_control(trial_store, control_store):
    trial = store_metrics[store_metrics['STORE_NBR'] == trial_store]
    control = store_metrics[store_metrics['STORE_NBR'] == control_store]
    # Split periods
    trial_pre = trial[trial['YEARMONTH'] < trial_start]
    trial_post = trial[(trial['YEARMONTH'] >= trial_start) & (trial['YEARMONTH'] <= trial_end)]
    control_pre = control[control['YEARMONTH'] < trial_start]
    control_post = control[(control['YEARMONTH'] >= trial_start) & (control['YEARMONTH'] <= trial_end)]
    # Sales comparison
    t_stat, p_value = ttest_ind(trial_post['TOTAL_SALES'], control_post['TOTAL_SALES'])
    print(f"\n=== Store {trial_store} vs Control {control_store} ===")
    print(f"P-value (Sales): {p_value}")
    # Driver analysis
    print("\nDriver Analysis:")
    print("Trial Avg Customers:", trial_post['N_CUSTOMERS'].mean())
    print("Control Avg Customers:", control_post['N_CUSTOMERS'].mean())
    print("Trial Txns/Customer:", trial_post['TXNS_PER_CUSTOMER'].mean())
    print("Control Txns/Customer:", control_post['TXNS_PER_CUSTOMER'].mean())

In [15]:
# 8. Run Comparison

for trial_store, control_store in control_stores.items():
    compare_trial_control(trial_store, control_store)


=== Store 77 vs Control 233.0 ===
P-value (Sales): 0.10314505225726589

Driver Analysis:
Trial Avg Customers: 47.333333333333336
Control Avg Customers: 38.333333333333336
Trial Txns/Customer: 1.0404255319148936
Control Txns/Customer: 1.0453703703703703

=== Store 86 vs Control 155.0 ===
P-value (Sales): 0.2265652364626469

Driver Analysis:
Trial Avg Customers: 109.0
Control Avg Customers: 96.0
Trial Txns/Customer: 1.2357036435053501
Control Txns/Customer: 1.261076611580531

=== Store 88 vs Control 125.0 ===
P-value (Sales): 0.025050459804835886

Driver Analysis:
Trial Avg Customers: 128.66666666666666
Control Avg Customers: 105.0
Trial Txns/Customer: 1.253563332530894
Control Txns/Customer: 1.170370607650382
